# 22 · Direct and self-coherent detection

An optical receiver turns light into an electrical current with a
**photodiode**.  A photodiode responds to the optical *power*, the number of
photons arriving per second, not to the electric field itself: it measures
the **intensity** `|x|²` and loses the phase.  There are two ways to build a
receiver around it:

```
 direct detection (IM-DD):     laser ─► intensity modulator ─► fiber ─► photodiode ─► ADC
                                                                          |x|²

 coherent detection:           laser ─► IQ modulator ─► fiber ─► 90° hybrid ─► balanced ─► ADC
                                                                   ▲            photodiodes
                                                        local oscillator         x · LO*
```

**Intensity modulation and direct detection** (IM-DD) puts the information
on the power alone: one photodiode, no second laser, the receiver of data
centre and access links.  A **coherent** receiver beats the signal with a
local oscillator (LO) and recovers the full field, amplitude and phase, as
in the rest of this course.  Between the two, **self-coherent** detection
sends a carrier along with the signal, so a single photodiode can recover
the field.

This notebook covers:

1. the **square law**: what a photodiode keeps and loses, and the bandwidth
   it doubles;
2. **IM-DD with unipolar PAM-4**: levels, eyes, the BER against received
   power, thermal and shot noise, and the laser's intensity noise;
3. **coherent detection**: why the LO makes the receiver so much more
   sensitive, and why its photodiodes come in balanced pairs;
4. **self-coherent detection** with a carrier, and its signal-signal beat.

It builds on the noise and metrics of
[06](06-awgn-and-performance-metrics.ipynb) and the optical fields of
[20](20-chromatic-dispersion.ipynb).  Samples are dimensionless: functions
that model absolute power take it as an argument (`received_power_dbm`,
`lo_power_dbm`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RC, RRC, Constellation, Rect, SmoothRect
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
WIDE = np.multiply(plt.rcParams["figure.figsize"], (2, 1))
C16 = Constellation.qam(16)
PAM4 = Constellation.pam(4, unipolar=True)
Q = 1.602176634e-19  # elementary charge [C]


def host(a):
    return to_device(a, "cpu")


def optical(sig):
    """Declare a field optical, on a carrier at 193.4 THz (1550 nm)."""
    return sig.replace(domain="OPT", center_frequency=193.4e12)


def intensity_modulate(sig):
    """The field whose power is the unipolar waveform `sig`."""
    xp = sig.xp
    field = xp.sqrt(xp.maximum(sig.samples, 0)).astype(xp.complex64)
    return optical(sig.replace(samples=field))

## 1. The square law

`apply_photodetection` models the diode.  It returns `|x|²` in the input's
power units plus the noise of the photocurrent: the optical power `P` gives
the current `I = R P`, with `R` the **responsivity** (about 0.8 A/W at 1550
nm), and the current carries **shot noise**, of variance `q I f_s` per
sample, and the receiver amplifier's **thermal noise**, `i_n² f_s / 2`.
`received_power_dbm` says what mean optical power the samples stand for.

### What the diode loses

The field is a complex number; its intensity is a real, non-negative one.
Everything carried by the phase disappears.  Below, a QPSK field
(raised-cosine pulses, rolloff 0.5, so that the field takes the symbol
values exactly at the symbol instants) and its intensity: the four QPSK symbols all have the
same power, so at the symbol instants (dots) the intensity is the same for
every symbol.  The dips between symbols only say that the phase changed,
not to what.  A photodiode cannot receive QPSK; IM-DD uses formats that
differ in power, such as PAM.

In [ ]:
qpsk = ck.generate(Constellation.psk(4), num_symbols=64, symbol_rate=10e9, sps=16,
                   pulse=RC(rolloff=0.5), rng=1, device=DEVICE)  # fmt: skip
x = host(qpsk.samples[16 * 20 : 16 * 36])
x = x / np.abs(x[0])  # unit power at the symbol instants
t = np.arange(x.size) / 16
fig, axes = plt.subplots(2, 1, figsize=WIDE * (1, 1.1), sharex=True)
axes[0].plot(t, x.real, label="I")
axes[0].plot(t, x.imag, label="Q")
axes[0].plot(t[::16], x.real[::16], "o", color="C0")
axes[0].plot(t[::16], x.imag[::16], "o", color="C1")
axes[0].set_ylabel("Field")
axes[0].set_title("QPSK: the information is in the phase")
axes[0].legend(loc="upper right")
axes[1].plot(t, np.abs(x) ** 2, color="C2")
axes[1].plot(t[::16], np.abs(x[::16]) ** 2, "o", color="C2")
axes[1].set_ylabel("Intensity |x|²")
axes[1].set_xlabel("Time [symbol periods]")
axes[1].set_title("What the photodiode sees: the same power for every symbol")
plt.show()

### The bandwidth doubles

Squaring multiplies the field by itself, so its spectrum is convolved with
itself: a field of band `±(1 + ρ) R_s / 2` gives an intensity of band
`±(1 + ρ) R_s`, plus a line at 0 Hz, the mean power.  Below, 16-QAM with
rolloff 0.5 at 4 samples per symbol: the field and its intensity (mean
removed).

At 2 samples per symbol the field fits in the sampled band `±R_s`, but its
intensity does not, and the part beyond `±R_s` folds back onto the signal.
The table gives the share of the intensity's power beyond `±R_s` for three
rolloffs.

In [ ]:
RS = 10e9
fig, ax = plt.subplots(figsize=WIDE * (1, 0.8))
for rolloff in (0.1, 0.5, 1.0):
    baseband = ck.generate(C16, num_symbols=2**15, symbol_rate=RS, sps=4,
                           pulse=RRC(rolloff=rolloff), rng=1, device=DEVICE)  # fmt: skip
    current = ck.impairments.apply_photodetection(
        optical(baseband), received_power_dbm=0.0, shot_noise=False
    )
    y = host(current.samples)
    spectrum = np.abs(np.fft.fft(y - y.mean())) ** 2
    f = np.fft.fftfreq(y.size, 1 / current.sampling_rate)
    beyond = spectrum[np.abs(f) > RS].sum() / spectrum.sum()
    print(f"rolloff {rolloff}: {100 * beyond:5.2f} % of the intensity beyond ±Rs "
          f"({10 * np.log10(beyond):.0f} dB)")  # fmt: skip
    if rolloff == 0.5:
        ck.plotting.plot_psd(baseband, nperseg=1024, ax=ax, label="field x")
        ck.plotting.plot_psd(current.replace(samples=current.samples - current.samples.mean()),
                             nperseg=1024, ax=ax, label="intensity |x|² (mean removed)",
                             title="16-QAM, RRC 0.5: the intensity is twice as wide",
                             legend=True)  # fmt: skip
for edge in (-RS, RS):
    ax.axvline(edge, linestyle=":", color="0.6")
plt.show()

The intensity is real, so its spectrum is symmetric and shown at positive
frequencies only; it reaches `(1 + ρ) R_s = 15 GHz`, and the line at 10 GHz
is a tone at the symbol rate.  The dotted lines are `±R_s`, the band a 2-sample-per-symbol record holds.
Only the intensity's tails lie beyond them, but they alias: a floor of
-31, -17 and -14 dB below the signal.  A direct-detection receiver filters
the photocurrent or samples it at `2 (1 + ρ)` samples per symbol when that
floor matters.

## 2. IM-DD with unipolar PAM-4

Intensity modulation puts the information on the optical power: four
levels `0, a, 2a, 3a`, a unipolar PAM-4 (`Constellation.pam(4, unipolar=True)`).
The modulator shapes the *power*, so the transmitted field is its square
root: the field amplitudes `0, 0.58, 0.82, 1` (relative to the top level)
are not equally spaced, but after the photodiode the levels are again.

Below, 50 GBd PAM-4 with smoothed NRZ pulses (`SmoothRect`), the power
levels the modulator sets, and the field amplitude that travels in the
fiber.

In [ ]:
RS = 50e9
pam_wave = ck.generate(PAM4, num_symbols=2**13, symbol_rate=RS, sps=8,
                       pulse=SmoothRect(rise_time=0.35), rng=2, device=DEVICE)  # fmt: skip
p = host(pam_wave.samples[8 * 40 : 8 * 60])
p = p / p.max()
t = np.arange(p.size) / 8
fig, axes = plt.subplots(1, 2, figsize=WIDE)
axes[0].plot(t, p, color="C0")
axes[1].plot(t, np.sqrt(np.maximum(p, 0)), color="C1")
for level in (0, 1 / 3, 2 / 3, 1):
    axes[0].axhline(level, color="0.5", linestyle=":", linewidth=0.8)
    axes[1].axhline(np.sqrt(level), color="0.5", linestyle=":", linewidth=0.8)
axes[0].set_title("Optical power: four equally spaced levels")
axes[1].set_title("Field amplitude: their square roots")
axes[0].set_ylabel("Power (relative to top level)")
axes[1].set_ylabel("Field amplitude")
for ax in axes:
    ax.set_xlabel("Time [symbol periods]")
plt.show()

### Eyes at the receiver

The photodiode returns the power levels, plus noise.  A receiver amplifier
(a transimpedance amplifier, TIA) with an input noise of `i_n = 20 pA/√Hz`
sets a fixed amount of **thermal noise** current; whether it matters
depends on how much current the light makes.  Below, the eye diagram after
the photodiode and the matched filter at three received powers, with the
symbol instant in the middle: at -8 dBm the four levels are clean, at -14 dBm the
noise fills a good part of the three eyes, at -20 dBm they are closed.

In [ ]:
I_N, RESP = 20e-12, 0.8  # thermal noise [A/sqrt(Hz)], responsivity [A/W]
field8 = intensity_modulate(pam_wave)
fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (2.4, 0.8))
)
for ax, power in zip(axes, (-8.0, -14.0, -20.0), strict=True):
    current = ck.impairments.apply_photodetection(
        field8, received_power_dbm=power, responsivity=RESP,
        thermal_noise_density=I_N, rng=3,
    )  # fmt: skip
    ck.plotting.plot_eye_diagram(ck.filtering.matched_filter(current), ax=ax,
                                 title=f"{power:.0f} dBm")  # fmt: skip
plt.show()

### BER against received power

To count errors, the same link at one sample per symbol, with plain NRZ
pulses (`Rect`): each sample is a level plus noise, the noise bandwidth is
`R_s / 2`, and the BER can be compared with theory.  With thermal noise
alone the noise is Gaussian with a fixed variance `σ² = i_n² R_s / 2` (in
A²), and `awgn_ber` gives the BER at the Es/N0 the photocurrent sets: for
real samples `Es/N0 = E[I²] / (2σ²)`, with `I` the current of each level.

The horizontal line is the threshold of the KP4 Reed-Solomon code used by
IM-DD Ethernet ([23](23-fec-in-optical-standards.ipynb)): below a BER of
2.4·10⁻⁴ it corrects everything.  The received power where the BER crosses
it is the receiver's **sensitivity**.

In [ ]:
tx = ck.generate(PAM4, num_symbols=2**17, symbol_rate=RS, sps=1, pulse=Rect(),
                 rng=4, device=DEVICE)  # fmt: skip
field1 = intensity_modulate(tx)
levels = host(tx.samples)


def thermal_ber(power_dbm):
    """BER of thermal-noise-limited PAM-4: the level currents against
    sigma^2 = i_n^2 Rs / 2, so Es/N0 = E[I^2] / (i_n^2 Rs)."""
    amps_per_unit = RESP * 1e-3 * 10 ** (np.asarray(power_dbm) / 10) / levels.mean()
    es = amps_per_unit**2 * np.mean(levels**2)
    return ck.metrics.awgn_ber(PAM4, esn0_db=10 * np.log10(es / (I_N**2 * RS)))


def sensitivity(powers, ber, threshold=2.4e-4):
    """Received power where a BER curve first crosses the threshold."""
    log_ber = np.log10(np.maximum(ber, 1e-12))
    i = int(np.argmax(log_ber < np.log10(threshold)))
    t = (np.log10(threshold) - log_ber[i - 1]) / (log_ber[i] - log_ber[i - 1])
    return powers[i - 1] + t * (powers[i] - powers[i - 1])


powers = np.arange(-18.0, -11.0, 1.0)
ber_thermal, ber_shot = [], []
for power in powers:
    for out, shot in ((ber_thermal, False), (ber_shot, True)):
        rx = ck.impairments.apply_photodetection(
            field1, received_power_dbm=power, responsivity=RESP,
            thermal_noise_density=I_N, shot_noise=shot, rng=5,
        )  # fmt: skip
        out.append(ck.metrics.ber(rx))
fine_p = np.linspace(-18.0, -10.0, 81)
theory = thermal_ber(fine_p)
s_theory = sensitivity(fine_p, theory)

fig, ax = plt.subplots()
ax.semilogy(fine_p, theory, color="0.6", label="theory, thermal noise")
ax.semilogy(powers, np.where(np.array(ber_thermal) > 0, ber_thermal, np.nan), "o",
            label="thermal noise")  # fmt: skip
ax.semilogy(powers, np.where(np.array(ber_shot) > 0, ber_shot, np.nan), "x",
            label="thermal + shot noise")  # fmt: skip
ax.axhline(2.4e-4, color="C3", linestyle=":", label="KP4 threshold")
ax.axvline(s_theory, color="C3", linestyle=":")
ax.set_xlabel("Received power [dBm]")
ax.set_ylabel("BER")
ax.set_ylim(1e-7, 0.3)
ax.set_title("PAM-4 IM-DD at 50 GBd, 20 pA/√Hz")
ax.legend()
plt.show()
print(f"sensitivity at the KP4 threshold: {s_theory:.1f} dBm (theory), "
      f"{sensitivity(powers, ber_thermal):.1f} dBm (simulated)")  # fmt: skip

The thermal-noise points follow the closed form, and the sensitivity is
about -14 dBm, 40 µW.  Shot noise costs little here: at these powers the thermal
term dominates.  The curve is steep because the current, not the field,
carries the signal: 1 dB more optical power is 2 dB more electrical SNR.

### Thermal against shot noise

Thermal noise belongs to the amplifier and is the same on every level.
Shot noise comes from the light itself, the random arrival of photons, and
its variance `q I f_s` grows with the current: the dark level `0` carries
none, the top level the most.  Below, histograms of the received samples
for a thermal-noise-limited receiver (-12 dBm, 20 pA/√Hz) and for one
without thermal noise at -30 dBm, where shot noise is all there is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=WIDE)
for ax, (title, power, i_n, shot) in zip(axes, (
    ("thermal noise, -12 dBm", -12.0, I_N, False),
    ("shot noise only, -30 dBm", -30.0, 0.0, True),
), strict=True):  # fmt: skip
    rx = ck.impairments.apply_photodetection(
        field1, received_power_dbm=power, responsivity=RESP,
        thermal_noise_density=i_n, shot_noise=shot, rng=6,
    )  # fmt: skip
    y = host(rx.samples)
    counts, _, _ = ax.hist(y, bins=300, color="C0")
    spreads = [np.std(y[levels == lv]) for lv in np.unique(levels)]
    ax.set_ylim(
        0, 1.3 * counts[counts.size // 6 :].max()
    )  # the dark level may go off-scale
    ax.set_title(title)
    ax.set_xlabel("Received sample (power units of the input)")
    print(f"{title:26s} rms noise per level: " + ", ".join(f"{s:.3f}" for s in spreads))
axes[0].set_ylabel("Count")
plt.show()

With thermal noise the four peaks have the same width.  With shot noise
the widths grow as the square root of the level: zero at the dark level
(its peak runs off the top of the plot), `√2` and `√3` times the second
level's on the third and fourth.  The top
eye is the narrowest, so shot-noise-limited receivers (with an optical or
avalanche gain in front) space the levels unequally.

### The laser's intensity noise

The laser's power is not perfectly constant either: its **relative
intensity noise** (RIN), in dB/Hz, is a fluctuation proportional to the
power (`apply_intensity_noise`).  More received power does not help
against it: signal and noise grow together.  It sets a **BER floor**.
Directly modulated lasers, common in IM-DD, have RIN around -130 dB/Hz.

In [ ]:
fig, ax = plt.subplots()
powers_rin = np.arange(-18.0, -1.0, 2.0)
ax.semilogy(fine_p, theory, color="0.6", label="no RIN (theory)")
print(f"no RIN:         sensitivity {s_theory:5.1f} dBm")
for rin, marker in ((-135.0, "o"), (-130.0, "s"), (-127.0, "^")):
    noisy = ck.impairments.apply_intensity_noise(field1, rin_db_hz=rin, rng=7)
    ber = [
        ck.metrics.ber(ck.impairments.apply_photodetection(
            noisy, received_power_dbm=p, responsivity=RESP,
            thermal_noise_density=I_N, rng=8,
        ))
        for p in powers_rin
    ]  # fmt: skip
    ax.semilogy(powers_rin, np.where(np.array(ber) > 0, ber, np.nan), marker + "-",
                label=f"RIN {rin:.0f} dB/Hz")  # fmt: skip
    reached = min(ber) < 2.4e-4
    print(f"RIN {rin:.0f} dB/Hz: sensitivity "
          + (f"{sensitivity(powers_rin, ber):5.1f} dBm" if reached else "  never")
          + f", BER at -2 dBm {ber[-1]:.1e}")  # fmt: skip
ax.axhline(2.4e-4, color="C3", linestyle=":", label="KP4 threshold")
ax.set_xlabel("Received power [dBm]")
ax.set_ylabel("BER")
ax.set_ylim(1e-6, 0.3)
ax.legend()
plt.show()

At low power the thermal noise dominates and all curves agree; at high
power they flatten at a floor that only a quieter laser lowers.  At -135
dB/Hz the floor lies far below the KP4 threshold and costs a fraction of a
dB; at -130 dB/Hz it is close enough to cost almost 3 dB of sensitivity; at
-127 dB/Hz the link never reaches the threshold at any power.  The RIN
noise is largest on the top level, like shot noise, since it is
proportional to the power.

## 3. Coherent detection: the LO as an amplifier

The IM-DD receiver needs tens of microwatts because its current, `R P`, must
stand above the amplifier's thermal noise.  A coherent receiver adds a
strong local oscillator of power `P_LO` (milliwatts) before the photodiode.
The beat between signal and LO gives a current proportional to
`R √(P_s P_LO)`: the LO amplifies the signal, noiselessly as far as the
amplifier is concerned.  The signal-to-noise ratio is

$$\frac{1}{\mathrm{SNR}} = \frac{f_s\,\big(q R (P_s + P_\mathrm{LO}) + i_n^2\big)}{R^2 P_s P_\mathrm{LO}}.$$

With a weak LO the thermal term `i_n²` dominates, and every dB of LO power
is a dB of SNR.  With a strong one the shot noise of the LO's own light,
`q R P_LO`, takes over, and the SNR stops at the **shot-noise limit**
`R P_s / (q f_s)`, set only by how many photons each symbol carries.
`apply_coherent_detection` models the 90° hybrid and two balanced photodiode
pairs.  Below, 16-QAM at 32 GBd received at -30 dBm, one microwatt, a
thousand times less than the IM-DD sensitivity above.

In [ ]:
RS_COH = 32e9
P_S = -30.0  # dBm
qam = optical(
    ck.generate(C16, num_symbols=2**15, symbol_rate=RS_COH, rng=9, device=DEVICE)
)
lo_powers = np.arange(-15.0, 16.0, 3.0)
ps, plo = 1e-3 * 10 ** (P_S / 10), 1e-3 * 10 ** (lo_powers / 10)
I_N_COH = 15e-12
snr_theory = 10 * np.log10(ps * plo / (RS_COH * (Q * (ps + plo) + I_N_COH**2)))  # R = 1
snr_shot = 10 * np.log10(ps / (Q * RS_COH))


def coherent_snr(lo_dbm, **kwargs):
    rx = ck.impairments.apply_coherent_detection(
        qam, received_power_dbm=P_S, lo_power_dbm=lo_dbm,
        thermal_noise_density=I_N_COH, rng=10, **kwargs,
    )  # fmt: skip
    return ck.metrics.snr(rx)


fig, ax = plt.subplots(figsize=WIDE * (0.65, 1))
ax.plot(lo_powers, snr_theory, color="0.6", label="theory, balanced")
ax.axhline(snr_shot, color="0.6", linestyle=":", label="shot-noise limit")
ax.plot(lo_powers, [coherent_snr(p) for p in lo_powers], "o", label="balanced, ideal")
ax.plot(lo_powers, [coherent_snr(p, cmrr_db=20.0, lo_rin_db_hz=-140.0) for p in lo_powers],
        "s-", label="balanced, CMRR 20 dB, LO RIN -140 dB/Hz")  # fmt: skip
ax.plot(lo_powers, [coherent_snr(p, balanced=False, lo_rin_db_hz=-140.0) for p in lo_powers],
        "^-", label="single diode, LO RIN -140 dB/Hz")  # fmt: skip
ax.set_xlabel("LO power [dBm]")
ax.set_ylabel("SNR [dB]")
ax.set_title("16-QAM at 32 GBd, -30 dBm received, 15 pA/√Hz")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.show()
direct = 10 * np.log10(ps**2 / (I_N_COH**2 * RS_COH / 2))  # R = 1, no LO
print(
    f"shot-noise limit {snr_shot:.1f} dB; the same light on one diode, no LO: {direct:.1f} dB"
)

The ideal balanced receiver follows the formula: thermal-noise-limited
below about 0 dBm of LO, gaining a dB per dB, then levelling off at the
shot-noise limit, 22.9 dB here.  Without an LO, one microwatt on the same diode would
give an SNR below zero.

Why **balanced** pairs?  Each photodiode also sees `|LO|²` and `|s|²`
directly, and the LO's power fluctuates with its own RIN.  A balanced pair
subtracts two diodes that see these terms equally, so they cancel, and only
the beat, which has opposite signs on the two diodes, remains.  Real pairs
are not perfectly matched; their **common-mode rejection ratio** (CMRR)
says how much leaks through.  The leaking RIN grows with the LO power, so
with a CMRR of 20 dB the SNR peaks and then falls; a single diode, which
rejects nothing, loses most of the gain.

## 4. Self-coherent detection

A carrier sent with the signal acts as a local oscillator that travels
with it, from the same laser: no second laser, no frequency offset, one
photodiode.  With the field `c + s(t)` the photodiode sees

$$|c + s|^2 = |c|^2 + 2\,\mathrm{Re}\{c^* s\} + |s|^2 .$$

The first term is a constant.  The middle one is the field itself, up to a
known phase.  The last, the **signal-signal beat interference** (SSBI), is
what direct detection of `s` alone would give: it occupies `±(1 + ρ) R_s`
around DC.  If the signal sits right next to the carrier, its beat with the
carrier lands on top of the SSBI.  Shifted away from the carrier by a
**guard band**, the beat lands at an intermediate frequency, clear of it.
The **carrier-to-signal power ratio** (CSPR) sets how strong the wanted
term is against the SSBI, at the cost of power spent on the carrier.

Below, 16-QAM at 10 GBd, its band starting `guard` above the carrier, a
photodiode at -10 dBm with 10 pA/√Hz thermal noise, then DC removal, a
shift back to baseband and the matched filter.  The photocurrent spectra
first, with and without a guard band, at a CSPR of 10 dB.

In [ ]:
RS = 10e9
qam10 = ck.generate(C16, num_symbols=2**15, symbol_rate=RS, sps=8,
                    pulse=RRC(rolloff=0.1), rng=11, device=DEVICE)  # fmt: skip
HALF_BAND = 0.55 * RS


def self_coherent(cspr_db, guard):
    """Photocurrent of 16-QAM sent `guard` Hz above a carrier, and the
    symbols recovered from it."""
    xp = qam10.xp
    step = qam10.sampling_rate / qam10.samples.shape[-1]
    shift = round((guard + HALF_BAND) / step) * step  # on the FFT grid
    s = ck.spectral.shift_frequency(qam10, frequency=shift)
    carrier = float(
        np.sqrt(np.mean(np.abs(host(s.samples)) ** 2) * 10 ** (cspr_db / 10))
    )
    sent = optical(s.replace(samples=(s.samples + carrier).astype(xp.complex64)))
    current = ck.impairments.apply_photodetection(
        sent, received_power_dbm=-10.0, thermal_noise_density=10e-12, rng=12
    )
    ac = current.replace(
        samples=(current.samples - current.samples.mean()).astype(xp.complex64)
    )
    baseband = ck.spectral.shift_frequency(ac, frequency=-shift)
    ssbi = xp.abs(s.samples) ** 2  # what |s|^2 alone contributes, same units
    return (
        ac.replace(samples=ac.samples.real),
        ac.replace(samples=ssbi - ssbi.mean()),
        ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(baseband)),
    )


fig, axes = plt.subplots(1, 2, figsize=WIDE, sharey=True)
recovered = {}
for ax, guard in zip(axes, (0.0, 1.1 * RS), strict=True):
    current, ssbi, recovered[guard] = self_coherent(10.0, guard)
    ck.plotting.plot_psd(current, nperseg=2048, ax=ax, label="photocurrent")
    ck.plotting.plot_psd(ssbi, nperseg=2048, ax=ax, label="SSBI |s|² alone",
                         title=f"guard band {guard / RS:.1f} Rs", xlim=(-0.5e9, 30e9))  # fmt: skip
    ax.axvspan(
        guard, guard + 2 * HALF_BAND, color="C2", alpha=0.15, label="signal × carrier"
    )
    ax.legend(loc="upper right")
axes[1].set_ylabel("")
plt.show()

The shaded band is where the beat of signal and carrier lands; the second
curve is the SSBI, `|s|²` alone, from DC to `(1 + ρ) R_s`, here 15 dB below
the wanted beat.  Without a guard band the two overlap and the SSBI is
hidden under the signal; with a guard band of `1.1 R_s` the signal sits
beside it, and above the SSBI's band only thermal noise remains.  (The line
at 10 GHz is the SSBI's own clock tone at the symbol rate.)  After the
shift back and the matched filter:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=WIDE)
for ax, (guard, sym) in zip(axes, recovered.items(), strict=True):
    ck.plotting.plot_constellation(
        sym,
        ax=ax,
        title=f"guard {guard / RS:.1f} Rs: SNR {ck.metrics.snr(sym, num_skip_symbols=100):.1f} dB",
    )
plt.show()

The SSBI is not random noise: it is a function of the symbols themselves,
so the clusters it blurs are not round Gaussian blobs (the corner and
centre points become rings).  That is what digital SSBI cancellation
exploits: from a first estimate of `s`, compute `|s|²` and subtract it.

How the SNR depends on the CSPR, for three guard bands:

In [ ]:
csprs = np.arange(4.0, 17.0, 2.0)
fig, ax = plt.subplots()
for guard in (0.0, 0.55 * RS, 1.1 * RS):
    snr = [
        ck.metrics.snr(self_coherent(c, guard)[2], num_skip_symbols=100) for c in csprs
    ]
    ax.plot(csprs, snr, "o-", label=f"guard {guard / RS:.2f} Rs")
    print(
        f"guard {guard / RS:.2f} Rs: SNR " + ", ".join(f"{v:.1f}" for v in snr) + " dB"
    )
ax.set_xlabel("CSPR [dB]")
ax.set_ylabel("SNR [dB]")
ax.set_title("Self-coherent 16-QAM, -10 dBm, 10 pA/√Hz")
ax.legend()
plt.show()

Without a guard band the SSBI falls on the signal, and only a stronger
carrier helps: the SNR rises with the CSPR.  A guard band as wide as the
signal's band moves the signal clear of it; then the limit is the thermal
noise, and at a fixed received power a stronger carrier leaves less power
for the signal, so the SNR falls with the CSPR.  Half a guard band sits in
between and levels off at a CSPR of 14-16 dB, where the remaining SSBI and
the thermal noise balance.  Practical receivers use a smaller guard
band with digital SSBI cancellation, or the Kramers-Kronig receiver, which
needs no guard band at all when the carrier is strong enough; the price of
the guard band is twice the electrical bandwidth.

**Summary**

* A photodiode measures `|x|²`: it loses the phase, and its output has
  twice the field's bandwidth.
* IM-DD of unipolar PAM-4 at short reach is limited by the receiver's
  thermal noise; its BER against received power follows `awgn_ber` at the
  Es/N0 the photocurrent gives.  Shot noise and laser RIN grow with the
  level, and RIN sets a floor that more power cannot lower.
* A coherent receiver's LO amplifies the signal above the thermal noise, up
  to the shot-noise limit: microwatts suffice where IM-DD needs tens of
  them.  Balanced photodiodes cancel the LO's intensity noise.
* A carrier sent with a QAM signal makes one photodiode a coherent
  receiver, traded against the signal-signal beat through the guard band
  and the CSPR.

**Next:** [23 · FEC in optical standards](23-fec-in-optical-standards.ipynb).